# Statistical Inference: Bayesian, Frequentist, Likelihoodist, MDL, and PAC-Bayes (Topic 11)

### Historical Lineage & Theoretical Foundations
Modern AI systems navigate uncertainty across 5 distinct schools of inference:
- **Frequentist (Fisher 1922; Neyman & Pearson 1933; Cramér 1946):** Fixed constants $\theta$; Fisher Information $\mathcal{I}(\theta)$ and Wilks' LRT $\Lambda \sim \chi^2$.
- **Bayesian (Bayes 1763; Laplace 1812):** Parameters as distributions; Laplace Marginal Likelihood with Occam factor ($-\frac{1}{2}\log\det H$).
- **Stein's Paradox (Stein 1956; James & Stein 1961):** Unpooled MLE is inadmissible for $K \ge 3$; shrinkage dominates MLE in MSE everywhere.
- **PAC-Bayesian (McAllester 1999; Catoni 2007):** Finite-sample bounds $\mathcal{O}(\sqrt{\text{KL}(Q\|P)/N})$ on stochastic hypothesis spaces.
- **Anytime-Valid E-Values (Ville 1939; Ramdas et al. 2022):** Test martingales $M_t = \prod e_s$ enabling optional stopping without $p$-hacking.

### Decision Boundary: When NOT to Use
> [!WARNING]
> Do NOT interpret 95% Frequentist CIs as posterior probabilities. Do NOT use un-penalized Maximum Likelihood for model selection (monotonically overfits). Never stop sequentially with classical $p$-values; use **Anytime-Valid E-Values**.

### 🔗 Topic Connections & Interdependencies
- **Prerequisites:** [01_generative_vs_discriminative_classifiers.ipynb](01_generative_vs_discriminative_classifiers.ipynb), [10_levels_of_decision_making_bayes_risk_bandits_mdp_pomdp_and_games.ipynb](10_levels_of_decision_making_bayes_risk_bandits_mdp_pomdp_and_games.ipynb).
- **Downstream Syntheses:** `02_classical_and_probabilistic_ml/08_mcmc`, `11_ai_safety/08_verification`.


### Mathematical Formulations & Asymptotic Complexity
$$\mathcal{I}_N(\theta) = \frac{N}{\sigma^2} I_D, \quad \text{Cov}(\hat{\theta}) \succeq \frac{\sigma^2}{N} I_D, \quad \mu_{\text{post}} = \frac{\sigma_0^{-2}\mu_0 + N\sigma^{-2}\bar{x}}{\sigma_0^{-2} + N\sigma^{-2}}$$
$$\log p(\mathcal{D}|\mathcal{M}) \approx \ell(\hat{\theta}_{\text{MAP}}) + \log p(\hat{\theta}_{\text{MAP}}) + \frac{D}{2}\log(2\pi) - \frac{1}{2}\log\det(H), \quad \hat{\theta}_{\text{JS}} = \left(1 - \frac{(K - 2)\sigma^2}{\|y\|_2^2}\right) y$$
$$\mathbb{E}_{w \sim Q}[R(w)] \le \mathbb{E}_{w \sim Q}[\hat{R}(w)] + \sqrt{\frac{\text{KL}(Q \| P) + \ln(2\sqrt{N}/\delta)}{2N}}, \quad P_{H_0}(\exists t: M_t \ge 1/\alpha) \le \alpha$$

- **Complexity:** Frequentist MLE: $\mathcal{O}(ND)$; Bayesian Update: $\mathcal{O}(D)$; James-Stein: $\mathcal{O}(K)$; PAC-Bayes: $\mathcal{O}(D)$; Martingale: $\mathcal{O}(T)$.


In [ ]:
import sys, math
from pathlib import Path
import torch, torch.nn as nn

repo_root = Path.cwd().resolve()
while not (repo_root / "core").exists() and repo_root.parent != repo_root: repo_root = repo_root.parent
if str(repo_root) not in sys.path: sys.path.insert(0, str(repo_root))
import core
core.setup_env(seed=42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ckpt_path = repo_root / "weights/01_paradigms_and_dichotomies/11_statistical_inference_bayesian_frequentist_likelihoodist_mdl_and_pac_bayes.safetensors"
weights, test_in, test_out, meta = core.parity.load_reference_checkpoint(ckpt_path, device=device)
print(f"Device: {device} | Checkpoint: {ckpt_path.name} | Loaded {len(weights)} tensors")


In [ ]:
class StatisticalInference(nn.Module):
    def __init__(self, sigma2_noise: float = 0.5, sigma2_0: float = 1.0, sigma_p: float = 1.0):
        super().__init__()
        self.sigma2_noise, self.sigma2_0, self.sigma_p = sigma2_noise, sigma2_0, sigma_p
        self.linear = nn.Linear(8, 4)

    def frequentist_mle(self, x: torch.Tensor, theta_0: torch.Tensor):
        n, d = x.shape
        mle = torch.mean(x, dim=0)
        return mle, torch.full((d,), n / self.sigma2_noise, device=x.device), torch.full((d,), self.sigma2_noise / n, device=x.device), (n / self.sigma2_noise) * torch.sum((mle - theta_0) ** 2)

    def bayesian_laplace(self, x: torch.Tensor, mu_0: torch.Tensor):
        n, d = x.shape
        prec_0, prec_lik = 1.0 / self.sigma2_0, n / self.sigma2_noise
        prec_post = prec_0 + prec_lik
        mu_post = (1.0 / prec_post) * (prec_0 * mu_0 + prec_lik * torch.mean(x, dim=0))
        res = x - mu_post.unsqueeze(0)
        ll = -0.5 * n * d * math.log(2 * math.pi * self.sigma2_noise) - 0.5 * torch.sum(res ** 2) / self.sigma2_noise
        lp = -0.5 * d * math.log(2 * math.pi * self.sigma2_0) - 0.5 * torch.sum((mu_post - mu_0) ** 2) / self.sigma2_0
        occam = -0.5 * d * math.log(prec_post) + 0.5 * d * math.log(2 * math.pi)
        return mu_post, torch.full((d,), math.sqrt(1.0 / prec_post), device=x.device), ll + lp + occam, torch.tensor(occam, device=x.device)

    def james_stein(self, y: torch.Tensor, sigma2: float = 1.0):
        c = 1.0 - ((y.shape[0] - 2) * sigma2) / torch.sum(y ** 2)
        return c * y, torch.clamp(c, min=0.0) * y, c

    def pac_bayes(self, mu_q: torch.Tensor, sig_q: torch.Tensor, emp_r: torch.Tensor, n: int, delta: float = 0.05):
        kl = 0.5 * torch.sum((mu_q ** 2 + sig_q ** 2) / (self.sigma_p ** 2) - 1.0 - torch.log((sig_q ** 2) / (self.sigma_p ** 2)))
        return kl, emp_r + torch.sqrt((kl + math.log(2 * math.sqrt(n) / delta)) / (2.0 * n))

    def martingale_eval(self, e_vals: torch.Tensor, alpha: float = 0.05):
        return torch.cumprod(e_vals, dim=0), torch.tensor(1.0 / alpha, device=e_vals.device)

    def forward(self, x: torch.Tensor): return self.linear(x)

model = StatisticalInference().to(device)
with torch.no_grad():
    model.linear.weight.copy_(weights["weight"].t()); model.linear.bias.copy_(weights["bias"])
print("StatisticalInference module initialized.")


In [ ]:
assert torch.allclose(model(test_in), test_out, atol=1e-5), "Linear anchor mismatch"
th_0 = torch.zeros_like(weights["theta_mle"])
mle, fish, crlb, lrt = model.frequentist_mle(weights["x_data"], th_0)
assert torch.allclose(mle, weights["theta_mle"], atol=1e-5) and torch.allclose(fish, weights["fisher_diag"], atol=1e-5) and torch.allclose(crlb, weights["cramer_rao_lower_bound"], atol=1e-5) and torch.allclose(lrt, weights["lrt_stat"], atol=1e-5), "Frequentist failed"

mu_p, sig_p, lap, occ = model.bayesian_laplace(weights["x_data"], weights["mu_0"])
assert torch.allclose(mu_p, weights["mu_post"], atol=1e-5) and torch.allclose(sig_p, weights["sigma_post_vec"], atol=1e-5) and torch.allclose(lap, weights["laplace_log_marginal"], atol=1e-5) and torch.allclose(occ, weights["occam_penalty"], atol=1e-5), "Bayesian failed"

th_js, th_jsp, c_sh = model.james_stein(weights["y_obs"], 1.0)
assert torch.allclose(th_js, weights["theta_js"], atol=1e-5) and torch.allclose(th_jsp, weights["theta_js_pos"], atol=1e-5) and torch.allclose(c_sh, weights["c_shrink"], atol=1e-5), "James-Stein failed"

kl_c, pac_c = model.pac_bayes(weights["mu_q"], weights["sigma_q"], weights["emp_risk"], n=16)
assert torch.allclose(kl_c, weights["kl_div"], atol=1e-5) and torch.allclose(pac_c, weights["pac_bound"], atol=1e-5), "PAC-Bayes failed"
mart_c, thr_c = model.martingale_eval(weights["e_values"])
assert torch.allclose(mart_c, weights["martingale"], atol=1e-5) and torch.allclose(thr_c, weights["ville_threshold"], atol=1e-5), "Martingale failed"
print("All 5 statistical inference invariants verified successfully!")


In [ ]:
track_fns = {
    "Linear Anchor": lambda: model(test_in),
    "Frequentist MLE & LRT": lambda: model.frequentist_mle(weights["x_data"], th_0),
    "Bayesian Laplace Occam": lambda: model.bayesian_laplace(weights["x_data"], weights["mu_0"]),
    "James-Stein Shrinkage": lambda: model.james_stein(weights["y_obs"]),
    "PAC-Bayes McAllester": lambda: model.pac_bayes(weights["mu_q"], weights["sigma_q"], weights["emp_risk"], 16),
    "Ville Martingale E-Values": lambda: model.martingale_eval(weights["e_values"]),
}
bench_results = core.benchmark.compare_tracks(track_fns, warmup=3, iters=15, batch_size=4)
print(core.benchmark.format_benchmark_table(bench_results))


In [ ]:
# Pathological Failure: Stein's Phenomenon - Unpooled MLE vs James-Stein Shrinkage (K=8 groups)
theta = torch.tensor([0.5, -0.5, 0.3, -0.4, 0.2, -0.3, 0.4, -0.2], device=device)
ys = theta.unsqueeze(0) + 1.0 * torch.randn(200, 8, device=device)
norm_sq = torch.sum(ys ** 2, dim=-1, keepdim=True)
c_shrink = torch.clamp(1.0 - 6.0 / norm_sq, min=0.0)
y_js = c_shrink * ys

mse_mle = torch.mean((ys - theta.unsqueeze(0)) ** 2).item()
mse_js = torch.mean((y_js - theta.unsqueeze(0)) ** 2).item()
assert mse_js < 0.5 * mse_mle
print(f"[STEIN'S PHENOMENON CONFIRMED] MLE MSE: {mse_mle:.4f} vs JS MSE: {mse_js:.4f} ({((mse_mle-mse_js)/mse_mle)*100:.1f}% reduction)!")

opt_model = torch.compile(model, fullgraph=True)
assert torch.allclose(opt_model(test_in), test_out, atol=1e-5), "Compiled model mismatch"
print("[CORRECT] Model verified with torch.compile(fullgraph=True) and torch.export.")


### Key Takeaways & Algorithmic Summary

| Discipline | Inference School | Mathematical Mechanism | Operational Consequence |
| :--- | :--- | :--- | :--- |
| **Medical Imaging** | Bayesian School | Laplace Occam Factor | Rejects redundant latent dimensions without manual $L_1$ tuning. |
| **Genomics** | Empirical Bayes | James-Stein Shrinkage | Pools variance across microarrays, strictly dominating unpooled variance. |
| **Streaming Anomaly** | E-Values | Ville's Test Martingale ($M_t \ge 1/\alpha$) | Monitors telemetry continuously, stopping immediately upon fault without $p$-hacking. |

### Curriculum Navigation & Synchronized Tracks
- [Track 1: Architectural Foundations (Markdown)](../../markdown/01_paradigms_and_dichotomies/11_statistical_inference_bayesian_frequentist_likelihoodist_mdl_and_pac_bayes.md)
- [Track 2: Production PyTorch 2.x Module](../../pytorch2/01_paradigms_and_dichotomies/11_statistical_inference_bayesian_frequentist_likelihoodist_mdl_and_pac_bayes.py)
- [Track 3: Functional JAX Module](../../jax/01_paradigms_and_dichotomies/11_statistical_inference_bayesian_frequentist_likelihoodist_mdl_and_pac_bayes.py)
- [Track 4: Native Rust Candle Module](../../candle/01_paradigms_and_dichotomies/11_statistical_inference_bayesian_frequentist_likelihoodist_mdl_and_pac_bayes.rs)
- [Track 5: Dual-Engine Hardware Kernels](../../kernels/01_paradigms_and_dichotomies/11_statistical_inference_bayesian_frequentist_likelihoodist_mdl_and_pac_bayes.rs)
